# Extent-rule arm with the new MTM extent law (Fusion_PhD-bhx7.14)

User, 2026-10-08: "ok go ahead and Rerun the GFTM extent-rule arm with the new law on the NSTX hypercubes".

**Arms.** `law2 rule` = `extent_rule_tf035_law2`: the `extent_rule_tf035` arm with only the extent law changed, from $\theta_{95} = 9.46\,k_y^{-0.351}\hat s^{-0.527}$ to the `bhx7.13` refit $16.30\,k_y^{-0.285}\hat s^{-0.593}q^{-0.355}$ ($\pi$, full span; one-sided extent = half). Per case NXGRID $=\arg\min_n|R(n)-\text{extent}/0.35\pi|$ on $[14,65]$, NBASIS $=$ NXGRID$-6$, WIDTH $=$ extent$/x_{max}$(NXGRID). Cases with $R>78.658$ are not built. Compared against `rule (9.46 law)` (`extent_rule_tf035`), `m11_new`, `rule s=1.41` (`extent_rule_tf035_s141`) and the max-basis arms (`extent_maxbasis_s141`, and `_s100` on n1000).

**Population and scoring** as `mtm_only_rescore` (`docs/mode_filtering.md`): GS2 $\gamma>10^{-3}$ and the classifier labels the GS2 dominant mode MTM. GFTM is never filtered; its dominant mode (argmax $\gamma$) is scored whatever its type. Capture = that mode has $T>0.15$ and $\omega<0$. A non-MTM dominant mode is a capture failure whose error still counts. No unstable mode is a MISS ($+\infty$).

**Pairing.** The headline table is on the cases every compared arm built (the paired set). A second table keeps the full MTM population and counts a case an arm could not build as a miss, as `mtm_only_rescore` did.

## Imports and settings

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binomtest
from general_analysis import mode_classification as mc
from general_analysis.mode_classification import T_TEAR

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
plt.rcParams.update({"axes.titlesize": 16, "axes.labelsize": 16, "xtick.labelsize": 14, "ytick.labelsize": 14, "legend.fontsize": 12})
load_dotenv(ROOT / "local.env", override=True)

analysis_name = "mtm_extent_rule_law2"
data_root = Path(os.environ["GK_DATA_ROOT"]).expanduser()
base = data_root / "GFTM/Runs/LATIN_HYPERCUBE/NSTX_MTM"
gs2_base = data_root / "GS2/Runs/LATIN_HYPERCUBE/NSTX_MTM"

arms_all = {   # label -> leaf
    "law2 rule": "extent_rule_tf035_law2",
    "rule (9.46 law)": "extent_rule_tf035",
    "m11_new": "m11_new",
    "rule s=1.41": "extent_rule_tf035_s141",
    "maxbasis s=1.41": "extent_maxbasis_s141",
    "maxbasis s=1": "extent_maxbasis_s100",
}
draws = {"n1000": ("beta_q_shat_ky_n1000", list(arms_all)),
         "n300":  ("beta_q_shat_ky_n300", [a for a in arms_all if a != "maxbasis s=1"])}
NEW, OLD, M11 = "law2 rule", "rule (9.46 law)", "m11_new"
gs2_unstable = 1e-3
n_boot, seed = 2000, 0

## Load

Per draw: the GS2 tail-averaged cube (`pyro_cube_avg`) and its classifier label, and every GFTM arm's dominant-mode $\gamma$, $\omega$, $T$ (pyro `FieldLine` with each sample's own $dl$, via `mc.tearing_parity`). NBASIS per sample: the GyroRun leaves carry it in `parameter_dict`, the legacy rule leaves as the `nbasis_used` coordinate (audited from `out.gftm.localdump`); `m11_new` is a fixed 24 for every case.

In [ ]:
def dom(ds, v):
    """Value of (sample, mode) array v at each sample's dominant (argmax gamma) mode."""
    g = np.asarray(ds.growth_rate.transpose("sample", "mode").values, float)
    k = np.where(np.isfinite(g), g, -np.inf).argmax(axis=1)
    return v[np.arange(len(k)), k]

def nbasis(scan, ds, leaf):
    if "NBASIS_MAX" in scan.parameter_dict:
        return np.asarray(scan.parameter_dict["NBASIS_MAX"], float)
    if "nbasis_used" in ds.coords:
        return np.asarray(ds.nbasis_used.values, float)
    assert leaf == "m11_new", leaf
    return np.full(ds.sizes["sample"], 24.0)   # registry gftm_m11_new: NBASIS 24 on every case

D = {}
for dname, (draw, arms) in draws.items():
    gscan, gs2 = mc.load_gs2_cube(gs2_base / draw, "pyro_cube_avg")
    names = [str(n) for n in gs2.sample_name.values]
    gamma_gs2 = np.asarray(gs2.growth_rate.values, float).reshape(len(names), -1)[:, 0]
    unstable = gamma_gs2 > gs2_unstable
    label = np.where(unstable, mc.classify(mc.indicators(gscan, gs2)), "").astype(object)
    rec = dict(names=names, gamma_gs2=gamma_gs2, unstable=unstable, label=label, mtm=unstable & (label == "MTM"), arm={},
               unit=f"{gs2.growth_rate.data.units:~P}")
    for arm in arms:
        ascan, ds = mc.load_gftm_cube(base / draw / arms_all[arm])
        row = {str(n): i for i, n in enumerate(ds.sample_name.values)}
        take = np.array([row.get(n, -1) for n in names])
        have, t = take >= 0, np.maximum(take, 0)
        g = np.asarray(ds.growth_rate.transpose("sample", "mode").values, float)
        om = np.asarray(ds.mode_frequency.transpose("sample", "mode").values, float)
        tp = mc.tearing_parity(ascan, ds, take[have])
        Tp = np.asarray(tp["T"].isel(kx=0, drop=True, missing_dims="ignore").transpose("sample", "mode").values, float)
        nb = nbasis(ascan, ds, arms_all[arm])
        a = dict(have=have, gamma=np.where(have, dom(ds, g)[t], np.nan), omega=np.where(have, dom(ds, om)[t], np.nan),
                 T=np.where(have, dom(ds, Tp)[t], np.nan), nbasis=np.where(have, nb[t], np.nan))
        a["mtm_mode"] = a["have"] & (a["gamma"] > 0) & (a["omega"] < 0) & (a["T"] > T_TEAR)
        if "NXGRID" in ascan.parameter_dict:
            a["nxgrid"] = np.where(have, np.asarray(ascan.parameter_dict["NXGRID"], float)[t], np.nan)
        rec["arm"][arm] = a
        print(dname, arm, "samples in cube", ds.sizes["sample"], "matched", int(have.sum()), "| gamma units GFTM", ds.growth_rate.data.units, "GS2", rec["unit"])
    D[dname] = rec
    lab = label[unstable]
    nm = int(rec["mtm"].sum())
    print(f"{dname}: {len(names)} cases | GS2-stable {len(names) - unstable.sum()} | KBM {int((lab == 'KBM').sum())} | "
          f"mixed {int((lab == 'mixed').sum())} | MTM kept {nm} | removed {len(names) - nm}")

## Built, infeasible and floored cases

Infeasible = no deck (the rule's $R>78.658$ wall). Floored = NXGRID at the floor 14 (finer $\theta_{first}$ than asked; the extent is still exact).

In [ ]:
for dname, r in D.items():
    n, m = len(r["names"]), r["mtm"]
    for arm in (NEW, OLD):
        a = r["arm"][arm]
        fl = (a["nxgrid"] == 14) if "nxgrid" in a else (r["arm"][arm]["nbasis"] == 8)   # NBASIS 8 <=> NXGRID 14
        print(f"{dname} {arm:16s}: infeasible {int((~a['have']).sum())} of {n} ({int((~a['have'] & m).sum())} in the MTM population); "
              f"floored {int((fl & a['have']).sum())} of {int(a['have'].sum())}; "
              f"NBASIS min/median/max {np.nanmin(a['nbasis']):.0f}/{np.nanmedian(a['nbasis']):.0f}/{np.nanmax(a['nbasis']):.0f}")
    both = ~r["arm"][NEW]["have"] & ~r["arm"][OLD]["have"]
    print(f"{dname}: infeasible in both {int(both.sum())}; new only {int((~r['arm'][NEW]['have'] & r['arm'][OLD]['have']).sum())}; "
          f"old only {int((r['arm'][NEW]['have'] & ~r['arm'][OLD]['have']).sum())}")

## Score

medlog = median $|\log_{10}(\gamma_{GFTM}/\gamma_{GS2})|$, medAE = median $|\gamma_{GFTM}-\gamma_{GS2}|$ (LogMed = $\log_{10}$ medAE), both on the dominant mode with misses $+\infty$. 95% bootstrap intervals on the same resampled cases for every arm. Paired: McNemar exact test (binomial on the discordant captures) of `law2 rule` against the old-law rule and against `m11_new`, and the bootstrap interval of the paired medlog difference.

In [ ]:
def errors(r, arm):
    a = r["arm"][arm]
    pos = a["have"] & (a["gamma"] > 0)
    with np.errstate(divide="ignore", invalid="ignore"):
        e = np.where(pos, np.abs(np.log10(a["gamma"] / r["gamma_gs2"])), np.inf)
    ae = np.where(pos, np.abs(a["gamma"] - r["gamma_gs2"]), np.inf)
    return e, ae, a["mtm_mode"], ~pos

rng = np.random.default_rng(seed)
def table(r, arms, mask):
    n = int(mask.sum())
    b = rng.integers(0, n, (n_boot, n))
    out = {}
    for arm in arms:
        e, ae, cap, miss = (x[mask] for x in errors(r, arm))
        out[arm] = dict(n=n, medlog=np.median(e), ci=np.percentile(np.median(e[b], axis=1), [2.5, 97.5]),
                        medae=np.median(ae), ae_ci=np.percentile(np.median(ae[b], axis=1), [2.5, 97.5]),
                        capture=cap.mean(), cap_ci=np.percentile(cap[b].mean(axis=1), [2.5, 97.5]),
                        capfail=int((~cap & ~miss).sum()), misses=int(miss.sum()),
                        nbasis=np.nanmedian(r["arm"][arm]["nbasis"][mask]))
    return out, b

def paired(r, x, y, mask, b):
    ex, _, cx, _ = (v[mask] for v in errors(r, x))
    ey, _, cy, _ = (v[mask] for v in errors(r, y))
    k01, k10 = int((~cx & cy).sum()), int((cx & ~cy).sum())   # y only, x only
    p = binomtest(min(k01, k10), k01 + k10, 0.5).pvalue if k01 + k10 else 1.0
    d = np.median(ex[b], axis=1) - np.median(ey[b], axis=1)
    return dict(x_only=k10, y_only=k01, p=p, dmed=np.median(ex) - np.median(ey), dci=np.percentile(d, [2.5, 97.5]))

TP, TF, PR = {}, {}, {}
for dname, r in D.items():
    arms = draws[dname][1]
    common = r["mtm"] & np.logical_and.reduce([r["arm"][a]["have"] for a in arms])
    TP[dname], b = table(r, arms, common)
    PR[dname] = {y: paired(r, NEW, y, common, b) for y in arms if y != NEW}
    TF[dname], _ = table(r, arms, r["mtm"])
    print(f"\n{dname}: PAIRED MTM-only n={int(common.sum())} (MTM population {int(r['mtm'].sum())}; dropped as unbuilt by some arm "
          f"{int(r['mtm'].sum() - common.sum())})")
    print(f"{'arm':16s} medlog [95% CI] | medAE [CI] | capture [CI] | cap.fail | misses | med NBASIS")
    for arm in arms:
        t = TP[dname][arm]
        print(f"{arm:16s} {t['medlog']:.3f} [{t['ci'][0]:.3f},{t['ci'][1]:.3f}] | {t['medae']:.4f} [{t['ae_ci'][0]:.4f},{t['ae_ci'][1]:.4f}] | "
              f"{t['capture']:.3f} [{t['cap_ci'][0]:.3f},{t['cap_ci'][1]:.3f}] | {t['capfail']:3d} | {t['misses']:3d} | {t['nbasis']:.0f}")
    print("law2 rule vs:  captured by law2 only / by the other only | McNemar p | medlog difference [95% CI]")
    for y, s in PR[dname].items():
        print(f"  {y:16s} {s['x_only']:3d} / {s['y_only']:3d} | p = {s['p']:.3g} | {s['dmed']:+.3f} [{s['dci'][0]:+.3f},{s['dci'][1]:+.3f}]")
    print(f"\n{dname}: FULL MTM-only n={int(r['mtm'].sum())}, unbuilt = miss")
    for arm in arms:
        t = TF[dname][arm]
        print(f"{arm:16s} medlog {t['medlog']:.3f} [{t['ci'][0]:.3f},{t['ci'][1]:.3f}] | medAE {t['medae']:.4f} | capture {t['capture']:.3f} "
              f"[{t['cap_ci'][0]:.3f},{t['cap_ci'][1]:.3f}] | misses {t['misses']}")

## Plots

Figure 1: per database, LogMed $|\gamma_{GFTM}-\gamma_{GS2}|$ and capture for every arm on the paired MTM set, with 95% bootstrap bars. Figure 2: the low-level result behind it, $\gamma_{GFTM}$ against $\gamma_{GS2}$ for the new-law and old-law rule on the paired MTM set, captured cases filled and capture failures open. Counts and definitions are in the README.

In [ ]:
out = ROOT / "Plots" / analysis_name
out.mkdir(parents=True, exist_ok=True)
fig1, ax = plt.subplots(2, 2, figsize=(13, 9), sharex="col")
for j, dname in enumerate(("n300", "n1000")):
    arms, t = draws[dname][1], TP[dname]
    x = np.arange(len(arms))
    lm = np.log10([t[a]["medae"] for a in arms])
    ax[0, j].errorbar(x, lm, yerr=np.abs(np.log10([t[a]["ae_ci"] for a in arms]).T - lm), ls="", marker="o")
    ax[1, j].errorbar(x, [t[a]["capture"] for a in arms],
                      yerr=np.abs(np.array([t[a]["cap_ci"] for a in arms]).T - [t[a]["capture"] for a in arms]), ls="", marker="o")
    ax[0, j].set_title(dname)
    ax[1, j].set_xticks(x, arms, rotation=30, ha="right")
ax[0, 0].set_ylabel(r"LogMed $|\gamma_{\rm GFTM}-\gamma_{\rm GS2}|$")
ax[1, 0].set_ylabel("MTM capture")
fig1.tight_layout()
plt.show()

In [ ]:
fig2, ax = plt.subplots(2, 2, figsize=(12, 11))
for i, dname in enumerate(("n300", "n1000")):
    r = D[dname]
    common = r["mtm"] & np.logical_and.reduce([r["arm"][a]["have"] for a in draws[dname][1]])
    g = r["gamma_gs2"][common]
    for j, arm in enumerate((OLD, NEW)):
        a = r["arm"][arm]
        y, cap = a["gamma"][common], a["mtm_mode"][common]
        ok = y > 0
        ax[i, j].scatter(g[ok & cap], y[ok & cap], s=12, label="MTM captured")
        ax[i, j].scatter(g[ok & ~cap], y[ok & ~cap], s=12, facecolors="none", edgecolors="C3", label="not captured")
        lim = [min(g.min(), y[ok].min()), max(g.max(), y[ok].max())]
        ax[i, j].plot(lim, lim, "k--", lw=1)
        ax[i, j].set(xscale="log", yscale="log", xlim=lim, ylim=lim, title=f"{dname}, {arm}",
                     xlabel=rf"$\gamma_{{\rm GS2}}$ [{r['unit']}]", ylabel=rf"$\gamma_{{\rm GFTM}}$ [{r['unit']}]")
ax[0, 0].legend()
fig2.tight_layout()
plt.show()

## Save

In [ ]:
fig1.savefig(out / "fig1_logmed_capture_paired.png")
fig2.savefig(out / "fig2_gamma_scatter_old_vs_new_law.png")

## Interpretation

_Written after the executed run; see the Fusion_PhD README `results/mtm_extent_rule_law2/README.md`._